In [2]:
import pandas as pd

nav = pd.read_csv(
    "../data/raw/02_nav_history.csv"
)

nav.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [3]:
print(nav.shape)
print(nav.head())
print(nav.isnull().sum())

(46000, 3)
   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692
amfi_code    0
date         0
nav          0
dtype: int64


In [4]:
nav['date'] = pd.to_datetime(
    nav['date']
)

In [5]:
nav = nav.sort_values(
    ['amfi_code','date']
)

In [6]:
nav = nav.drop_duplicates()

In [7]:
nav = nav[
    nav['nav'] > 0
]

In [8]:
nav['nav'] = (
    nav.groupby('amfi_code')['nav']
    .ffill()
)

In [9]:
nav['daily_return_pct'] = (
    nav.groupby('amfi_code')['nav']
    .pct_change()
) * 100

In [10]:
nav.to_csv(
    "../data/processed/clean_nav.csv",
    index=False
)

In [12]:
transactions = pd.read_csv(
    "../data/raw/08_investor_transactions.csv"
)

print(transactions.shape)
print(transactions.head())

(32778, 13)
  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              SIP        1834   
1   INV002952       2024-01-01     148567       Redemption      392882   
2   INV003420       2024-01-01     118636              SIP         912   
3   INV003436       2024-01-01     118634              SIP        1102   
4   INV004691       2024-01-01     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female                77.1   
1       Punjab   Amritsar       B30     18-25    Male                 7.1   
2      Haryana  Faridabad       B30     36-45    Male                47.2   
3  Maharashtra     Mumbai       T30     36-45  Female                54.4   
4        Delhi      Noida       T30     26-35    Male                14.5   

  payment_mode kyc_status  
0          UPI   Verified  
1       Cheque   Verifie

In [13]:
transactions['transaction_date'] = (
    pd.to_datetime(
        transactions['transaction_date']
    )
)

In [14]:
transactions['transaction_type'] = (
    transactions['transaction_type']
    .str.strip()
    .str.title()
)

In [15]:
transactions = transactions[
    transactions['amount_inr'] > 0
]

In [16]:
print(
transactions['kyc_status']
.value_counts()
)

kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64


In [17]:
transactions = (
    transactions.drop_duplicates()
)

In [18]:
transactions.to_csv(
    "../data/processed/clean_transactions.csv",
    index=False
)

In [22]:
performance = pd.read_csv(
    "../data/raw/07_scheme_performance.csv"
)

print(performance.shape)
print(performance.head())

(40, 19)
   amfi_code                                   scheme_name       fund_house  \
0     119551     SBI Bluechip Fund - Regular Plan - Growth  SBI Mutual Fund   
1     119552      SBI Bluechip Fund - Direct Plan - Growth  SBI Mutual Fund   
2     119598    SBI Small Cap Fund - Regular Plan - Growth  SBI Mutual Fund   
3     119599     SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund   
4     119120  SBI Magnum Gilt Fund - Regular Plan - Growth  SBI Mutual Fund   

    category     plan  return_1yr_pct  return_3yr_pct  return_5yr_pct  \
0  Large Cap  Regular           12.42           12.36           14.45   
1  Large Cap   Direct           15.25           11.30           14.23   
2  Small Cap  Regular           24.56           23.39           20.67   
3  Small Cap   Direct           20.59           23.14           21.82   
4       Gilt  Regular            5.34            6.07            5.43   

   benchmark_3yr_pct  alpha  beta  sharpe_ratio  sortino_ratio  \
0          

In [23]:
numeric_cols = [
'return_1yr_pct',
'return_3yr_pct',
'return_5yr_pct',
'alpha',
'beta',
'sharpe_ratio',
'sortino_ratio'
]

for col in numeric_cols:
    performance[col] = pd.to_numeric(
        performance[col],
        errors='coerce'
    )

In [24]:
negative_sharpe = performance[
    performance['sharpe_ratio'] < 0
]

print(negative_sharpe)

Empty DataFrame
Columns: [amfi_code, scheme_name, fund_house, category, plan, return_1yr_pct, return_3yr_pct, return_5yr_pct, benchmark_3yr_pct, alpha, beta, sharpe_ratio, sortino_ratio, std_dev_ann_pct, max_drawdown_pct, aum_crore, expense_ratio_pct, morningstar_rating, risk_grade]
Index: []


In [25]:
performance = performance[
performance['expense_ratio_pct']
.between(0.1,2.5)
]

In [26]:
performance.to_csv(
    "../data/processed/clean_performance.csv",
    index=False
)

In [33]:
from sqlalchemy import create_engine

engine = create_engine(
    'sqlite:///../db/bluestock_mf.db'
)

In [34]:
fund_master = pd.read_csv(
    "../data/raw/01_fund_master.csv"
)

fund_master.to_sql(
    'dim_fund',
    engine,
    if_exists='replace',
    index=False
)

nav.to_sql(
    'fact_nav',
    engine,
    if_exists='replace',
    index=False
)

transactions.to_sql(
    'fact_transactions',
    engine,
    if_exists='replace',
    index=False
)

performance.to_sql(
    'fact_performance',
    engine,
    if_exists='replace',
    index=False
)

40

In [36]:
import sqlite3

conn = sqlite3.connect(
    "../db/bluestock_mf.db"
)

tables = pd.read_sql(
"""
SELECT name
FROM sqlite_master
WHERE type='table'
""",
conn
)

tables

,name
0,dim_fund
1,fact_nav
2,fact_transactions
3,fact_performance
